# ₿ Bitcoin Puzzle #71 — RTX PRO 6000 Blackwell ile solo, rastgele GPU araması

Bu notebook her şeyi kendisi yapar: **Çalışma zamanı → Tümünü çalıştır** (Runtime → Run all) demen yeterli.

1. GPU'yu ve hedef adresin bakiyesini kontrol eder. Bulmaca çözülmüşse durur.
2. Açık kaynak [VanitySearch-Bitcrack](https://github.com/FixedPaul/VanitySearch-Bitcrack) (FixedPaul, GPLv3) aracını **bu GPU'nun mimarisi için** (Blackwell = `sm_120`) derler. Kendi `-random` modu var. Raporlanan hız: RTX 5090'da ~8.8 Gkey/s.
   * **Optimize yaması (`VS_OPTIMIZE`):** GPU çekirdeğine uygulanan küçük bir yama. sm_120 üzerinde orijinale göre anahtar başına **%10.8 daha az GPU çevrimi** harcıyor (Nsight Compute, sabit saat). RTX PRO 6000 gibi saat limitli kartlarda bu ≈ **+%12** hız demek. Güç limitli bir laptopta (RTX 5050, 80 W) duvar saatinde +%9.0 ölçüldü. Yama gömülüdür; `vanitysearch-opt.patch` dosyasıyla aynıdır. Öz-testi geçemezse notebook orijinal kaynağa döner.
3. **Öz-test:** Rastgele bir test anahtarı saklar ve aracın onu gerçekten bulduğunu doğrular (birkaç saniye). Geçemezse arama başlamaz.
4. **Arama:** `2^70 … 2^71-1` aralığında rastgele arar. Araç bir anahtar bildirdiğinde Python'da bağımsız olarak doğrulanır, sonra Drive'a ve `/content/FOUND_KEY.txt`'e yazılır. Araç çökerse otomatik olarak yeniden başlatılır. İstatistikler oturumlar arasında Drive'da birikir.

**Başlamadan önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir* menüsünden **RTX PRO 6000 (Blackwell)** GPU'yu seç. Drive izni istendiğinde onay ver.

---

### ⚠️ Gerçekçi beklenti (lütfen oku)

Aralıkta 2^70 ≈ **1.18 × 10²¹** anahtar var. Public key açıkta olmadığı için Kangaroo/BSGS kullanılamıyor; tek yol kaba kuvvet. Tek bir RTX PRO 6000 ile tahmini hız **~9–10 Gkey/s**:

| Süre | Bulma olasılığı (10 Gkey/s) |
|---|---|
| 1 saat | ≈ 1 / 33 milyon |
| 1 gün | ≈ 1 / 1.37 milyon |
| 1 ay | ≈ 1 / 45 bin |
| Tüm aralık | ≈ **3.700 yıl** |

Ödül 7.1 BTC. Bu ödülle bir günlük GPU çalışmasının beklenen değeri yaklaşık **520 satoshi** (~0.0000052 BTC). Bu, Colab'da bir günlük GPU ücretinin çok altında kalır. Bunu bir piyango gibi düşün ve kaybetmeyi göze alabileceğin kadar compute harca.

**Colab politikası:** Colab'ın SSS sayfası "kripto para madenciliği" ve "şifre kırma" gibi işleri yönetilen çalışma zamanlarında yasaklı faaliyetler arasında sayıyor. Bu iş bunlara çok yakın görülebilir, oturum kapatma veya hesap kısıtlaması riski var. Bu riski sen değerlendir. Notebook Colab'a özel değildir; NVIDIA GPU'lu ve CUDA'lı herhangi bir Linux Jupyter ortamında (ör. kiralık GPU sunucuları) aynen çalışır.

**Tek GPU = tek süreç:** Aynı GPU'da iki arama aynı anda çalıştırılmaz. Notebook GPU'nun tamamını tek araca verir.

In [ ]:
#@title ⚙️ 1) Ayarlar
TARGET_ADDRESS = "1PWo3JeB9jrGwfHDNpdGK54CRas7fsVzXU"  #@param {type:"string"}
PUZZLE_BITS    = 71         #@param {type:"integer"}
#@markdown **VanitySearch stratejisi:** `native` = aracın kendi `-random` modu · `blocks` = rastgele 2^BLOCK_BITS bloklar (Drive'a kaydedilir, tekrar taranmaz).
VS_STRATEGY    = "native"   #@param ["native", "blocks"]
BLOCK_BITS     = 42         #@param {type:"integer"}
USE_DRIVE      = True       #@param {type:"boolean"}
#@markdown **Optimize yaması:** VanitySearch GPU çekirdeğine hız yamasını uygula (öz-testi geçemezse otomatik kapanır).
VS_OPTIMIZE    = True       #@param {type:"boolean"}
AUTO_INSTALL_CUDA = True    #@param {type:"boolean"}

RANGE_BITS  = PUZZLE_BITS - 1                 # 71. bulmaca: anahtar 2^70 ... 2^71-1 aralığında
RANGE_START = 1 << RANGE_BITS
RANGE_END   = (1 << PUZZLE_BITS) - 1
RANGE_SIZE  = RANGE_END - RANGE_START + 1
assert PUZZLE_BITS >= 34, 'GPU araçları çok küçük aralıklarda çalışmaz; test için en az 34. bulmacayı kullan.'
if BLOCK_BITS > RANGE_BITS - 2:          # test için küçük bulmaca seçildiyse bloğu aralığa sığdır
    BLOCK_BITS = RANGE_BITS - 2
    print(f'ℹ️  Küçük aralık: BLOCK_BITS = {BLOCK_BITS} olarak ayarlandı.')
print(f'Hedef : {TARGET_ADDRESS}')
print(f'Aralık: {RANGE_START:x} : {RANGE_END:x}  (2^{RANGE_BITS} = {RANGE_SIZE:.3e} anahtar)')

In [ ]:
#@title 🧰 2) Yardımcı fonksiyonlar (secp256k1 doğrulama, süreç yönetimi, çıktı ayrıştırma)
import os, re, sys, json, time, math, glob, shutil, pty, select, signal, secrets, hashlib, subprocess, statistics, datetime

# ---------------------------------------------------------------------------
# Saf Python secp256k1 + adres üretimi. GPU aracı bir anahtar bulduğunda onu
# buradan BAĞIMSIZ olarak doğruluyoruz; sahte/yanlış pozitif asla "bulundu" sayılmaz.
# ---------------------------------------------------------------------------
_P = 2**256 - 2**32 - 977
_G = (0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
      0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8)

def _ec_add(a, b):
    if a is None: return b
    if b is None: return a
    if a[0] == b[0]:
        if (a[1] + b[1]) % _P == 0: return None
        l = 3 * a[0] * a[0] * pow(2 * a[1], -1, _P) % _P
    else:
        l = (b[1] - a[1]) * pow(b[0] - a[0], -1, _P) % _P
    x = (l * l - a[0] - b[0]) % _P
    return (x, (l * (a[0] - x) - a[1]) % _P)

def pubkey_compressed(k):
    r, q = None, _G
    while k:
        if k & 1: r = _ec_add(r, q)
        q = _ec_add(q, q); k >>= 1
    return bytes([2 + (r[1] & 1)]) + r[0].to_bytes(32, 'big')

def _rmd160_py(msg):
    # OpenSSL 3 bazı sistemlerde ripemd160'ı kapatıyor; saf Python yedek.
    rol = lambda x, n: ((x << n) | (x >> (32 - n))) & 0xFFFFFFFF
    F = [lambda x, y, z: x ^ y ^ z,
         lambda x, y, z: (x & y) | (~x & z),
         lambda x, y, z: (x | ~y) ^ z,
         lambda x, y, z: (x & z) | (y & ~z),
         lambda x, y, z: x ^ (y | ~z)]
    KL = [0x00000000, 0x5A827999, 0x6ED9EBA1, 0x8F1BBCDC, 0xA953FD4E]
    KR = [0x50A28BE6, 0x5C4DD124, 0x6D703EF3, 0x7A6D76E9, 0x00000000]
    RL = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15, 7,4,13,1,10,6,15,3,12,0,9,5,2,14,11,8,
          3,10,14,4,9,15,8,1,2,7,0,6,13,11,5,12, 1,9,11,10,0,8,12,4,13,3,7,15,14,5,6,2,
          4,0,5,9,7,12,2,10,14,1,3,8,11,6,15,13]
    RR = [5,14,7,0,9,2,11,4,13,6,15,8,1,10,3,12, 6,11,3,7,0,13,5,10,14,15,8,12,4,9,1,2,
          15,5,1,3,7,14,6,9,11,8,12,2,10,0,4,13, 8,6,4,1,3,11,15,0,5,12,2,13,9,7,10,14,
          12,15,10,4,1,5,8,7,6,2,13,14,0,3,9,11]
    SL = [11,14,15,12,5,8,7,9,11,13,14,15,6,7,9,8, 7,6,8,13,11,9,7,15,7,12,15,9,11,7,13,12,
          11,13,6,7,14,9,13,15,14,8,13,6,5,12,7,5, 11,12,14,15,14,15,9,8,9,14,5,6,8,6,5,12,
          9,15,5,11,6,8,13,12,5,12,13,14,11,8,5,6]
    SR = [8,9,9,11,13,15,15,5,7,7,8,11,14,14,12,6, 9,13,15,7,12,8,9,11,7,7,12,7,6,15,13,11,
          9,7,15,11,8,6,6,14,12,13,5,14,13,13,7,5, 15,5,8,11,14,14,6,14,6,9,12,9,12,5,15,8,
          8,5,12,9,12,5,14,6,8,13,6,5,15,13,11,11]
    h = [0x67452301, 0xEFCDAB89, 0x98BADCFE, 0x10325476, 0xC3D2E1F0]
    msg = bytes(msg)
    ml = len(msg) * 8
    msg += b'\x80' + b'\x00' * ((55 - len(msg)) % 64) + ml.to_bytes(8, 'little')
    for off in range(0, len(msg), 64):
        X = [int.from_bytes(msg[off + 4 * i: off + 4 * i + 4], 'little') for i in range(16)]
        al, bl, cl, dl, el = h
        ar, br, cr, dr, er = h
        for j in range(80):
            r = j // 16
            t = (rol((al + F[r](bl, cl, dl) + X[RL[j]] + KL[r]) & 0xFFFFFFFF, SL[j]) + el) & 0xFFFFFFFF
            al, el, dl, cl, bl = el, dl, rol(cl, 10), bl, t
            t = (rol((ar + F[4 - r](br, cr, dr) + X[RR[j]] + KR[r]) & 0xFFFFFFFF, SR[j]) + er) & 0xFFFFFFFF
            ar, er, dr, cr, br = er, dr, rol(cr, 10), br, t
        t = (h[1] + cl + dr) & 0xFFFFFFFF
        h[1] = (h[2] + dl + er) & 0xFFFFFFFF
        h[2] = (h[3] + el + ar) & 0xFFFFFFFF
        h[3] = (h[4] + al + br) & 0xFFFFFFFF
        h[4] = (h[0] + bl + cr) & 0xFFFFFFFF
        h[0] = t
    return b''.join(x.to_bytes(4, 'little') for x in h)

def ripemd160(b):
    try:
        return hashlib.new('ripemd160', b).digest()
    except Exception:
        return _rmd160_py(b)

_sha256d = lambda b: hashlib.sha256(hashlib.sha256(b).digest()).digest()
_B58 = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'

def b58check_encode(payload):
    b = payload + _sha256d(payload)[:4]
    n, s = int.from_bytes(b, 'big'), ''
    while n:
        n, r = divmod(n, 58); s = _B58[r] + s
    return '1' * (len(b) - len(b.lstrip(b'\0'))) + s

def b58check_decode(s):
    n = 0
    for c in s: n = n * 58 + _B58.index(c)
    raw = b'\0' * (len(s) - len(s.lstrip('1'))) + n.to_bytes((n.bit_length() + 7) // 8, 'big')
    data, chk = raw[:-4], raw[-4:]
    if _sha256d(data)[:4] != chk: raise ValueError('base58 checksum hatalı')
    return data

def addr_from_key(k):  return b58check_encode(b'\x00' + ripemd160(hashlib.sha256(pubkey_compressed(k)).digest()))
def wif_from_key(k):   return b58check_encode(b'\x80' + k.to_bytes(32, 'big') + b'\x01')
def key_from_wif(w):   return int.from_bytes(b58check_decode(w)[1:33], 'big')

# Bilinen test vektörleri (özel anahtar = 1)
assert _rmd160_py(b'abc').hex() == '8eb208f7e05d987a9b044a8e98c6b087f15a0bfc'
assert addr_from_key(1) == '1BgGZ9tcN4rm9KBzDn7KprQz87SZ26SAMH'
assert wif_from_key(1) == 'KwDiBf89QgGbjEhKnhXJuH7LrciVrZi3qYjgd9M7rFU73sVHnoWn'

# ---------------------------------------------------------------------------
# Kabuk ve süreç yönetimi
# ---------------------------------------------------------------------------
def sh(cmd, check=True, quiet=False, tail=3000):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, executable='/bin/bash',
                       stdin=subprocess.DEVNULL)
    if not quiet or (check and r.returncode):
        out = (r.stdout + r.stderr).strip()
        if out: print(out[-tail:])
    if check and r.returncode:
        raise RuntimeError(f'Komut başarısız (rc={r.returncode}): {cmd}')
    return r.stdout

def _stop_proc(p):
    if p.poll() is not None: return
    for sig, wait in ((signal.SIGINT, 5), (signal.SIGKILL, 5)):
        try: os.killpg(p.pid, sig)
        except ProcessLookupError: return
        try: p.wait(wait); return
        except subprocess.TimeoutExpired: pass

def stream(cmd, cwd=None, timeout=None, info=None):
    """Aracı sahte terminal (pty) altında çalıştırır: çıktı tamponlanmaz, her satır anında gelir.
    Her satırı (\\r ve \\n ile bölünmüş) yield eder; çıktı yokken ~1 sn'de bir '' (nabız) yield eder.
    Döngüden çıkılınca (break / hata / Ctrl+C) süreç grubu öldürülür."""
    master, slave = pty.openpty()
    p = subprocess.Popen(cmd, cwd=cwd, stdin=subprocess.DEVNULL, stdout=slave, stderr=slave,
                         start_new_session=True, close_fds=True)
    os.close(slave)
    buf, t_end = b'', (time.time() + timeout) if timeout else None
    try:
        while True:
            if t_end and time.time() > t_end: break
            r, _, _ = select.select([master], [], [], 1.0)
            if not r:
                if p.poll() is not None: break
                yield ''
                continue
            try: data = os.read(master, 65536)
            except OSError: data = b''
            if not data: break
            buf += data
            *lines, buf = re.split(rb'[\r\n]+', buf)
            for ln in lines:
                s = ln.decode('utf-8', 'replace').strip()
                if s: yield s
        if buf.strip(): yield buf.decode('utf-8', 'replace').strip()
    finally:
        _stop_proc(p)
        os.close(master)
        if info is not None: info['rc'] = p.returncode

# ---------------------------------------------------------------------------
# VanitySearch-Bitcrack çıktısını ayrıştırma
#   "8812.3 MK/s - 1234 BKeys - 2^40.1 [..] ..."   /  "Priv (HEX): 0x...."
#   "  CandKey: ..."  (notebook'un yamasıyla: aracın kendi CPU kontrolü başarısız olsa bile aday anahtar)
# WIF satırları bilerek ayrıştırılmıyor: VanitySearch'ün CPU SHA-256'sı bazı derleyicilerle hatalı checksum üretiyor.
# ---------------------------------------------------------------------------
_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
_RX = {
    'vs_hex':   re.compile(r'Priv \(HEX\):\s*0x\s*([0-9A-Fa-f]+)'),
    'vs_cand':  re.compile(r'CandKey:\s*([0-9A-Fa-f]+)'),
    'vs_speed': re.compile(r'([\d.]+)\s*MK/s\s*-\s*([\d.]+)\s*BKeys'),
}

def parse_line(line):
    """('keys', [aday anahtarlar]) | ('speed', Mkeys/s, bu süreçte taranan toplam anahtar) | None"""
    m = _RX['vs_hex'].search(line)
    if m: return ('keys', [int(m.group(1), 16)])
    m = _RX['vs_cand'].search(line)
    if m:
        c = int(m.group(1), 16) % _N           # araç negatifini de denemiş olabilir: ikisini de aday say
        return ('keys', [c, (_N - c) % _N])
    m = _RX['vs_speed'].search(line)
    if m: return ('speed', float(m.group(1)), int(float(m.group(2)) * 1e9))
    return None

def fmt_num(x):
    for unit, d in (('E', 1e18), ('P', 1e15), ('T', 1e12), ('G', 1e9), ('M', 1e6), ('K', 1e3)):
        if abs(x) >= d: return f'{x / d:.2f} {unit}'
    return f'{x:.0f}'

def fmt_dur(s):
    s = int(s)
    d, s = divmod(s, 86400); h, s = divmod(s, 3600); m, s = divmod(s, 60)
    return (f'{d}g ' if d else '') + f'{h:02d}:{m:02d}:{s:02d}'

print('✅ Yardımcılar yüklendi, secp256k1/RIPEMD160/Base58 test vektörleri geçti.')

In [ ]:
#@title 🖥️ 3) GPU, hedef adres, bakiye ve kayıt klasörü kontrolü
import urllib.request
from IPython.display import display, HTML, Javascript

FOUND_KEY = None      # başka bir bulmacadan (ör. test) kalan "bulundu" durumunu sıfırla

# --- GPU ---
q = sh('nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader', quiet=True)
GPU_NAME, GPU_CC, GPU_MEM, GPU_DRIVER = [x.strip() for x in q.strip().splitlines()[0].split(',')]
CC = int(GPU_CC.replace('.', ''))
print(f'GPU    : {GPU_NAME}  (compute {GPU_CC}, {GPU_MEM}, driver {GPU_DRIVER})')
if CC < 120:
    print('⚠️  Bu bir Blackwell GPU değil. Notebook yine çalışır ama hız beklenenden düşük olur.')

# --- Hedef adres ---
_payload = b58check_decode(TARGET_ADDRESS)
assert len(_payload) == 21 and _payload[0] == 0, 'Hedef bir P2PKH (1...) adresi olmalı'
TARGET_H160 = _payload[1:].hex()
print(f'Hedef  : {TARGET_ADDRESS}  hash160={TARGET_H160}')

# --- Bakiye: bulmaca hâlâ açık mı? (çözülmüşse boşuna GPU yakmayalım) ---
def address_info(addr):
    last = None
    for base in ('https://mempool.space/api', 'https://blockstream.info/api'):
        try:
            req = urllib.request.Request(f'{base}/address/{addr}', headers={'User-Agent': 'puzzle-notebook'})
            with urllib.request.urlopen(req, timeout=20) as r:
                d = json.load(r)
            cs = d['chain_stats']
            return (cs['funded_txo_sum'] - cs['spent_txo_sum']) / 1e8, cs['spent_txo_count']
        except Exception as e:
            last = e
    print('⚠️  Bakiye sorgulanamadı:', last)
    return None, None

BALANCE, SPENT_COUNT = address_info(TARGET_ADDRESS)
if BALANCE is not None:
    print(f'Bakiye : {BALANCE:.8f} BTC')
    if BALANCE == 0 and PUZZLE_BITS > 50:
        raise RuntimeError('❌ Adreste bakiye yok: bulmaca zaten çözülmüş. Çalıştırmayı durdurdum.')
    if BALANCE == 0:
        print('ℹ️  Test: bu bulmaca zaten çözülmüş, sadece deneme amaçlı çalıştırılıyor.')
    if SPENT_COUNT:
        print('⚠️  Bu adresten harcama yapılmış → public key açıkta. Bu durumda brute-force yerine Kangaroo/BSGS çok daha hızlıdır.')

# --- Kayıt klasörü (Drive varsa oraya: oturum koparsa bulunan anahtar kaybolmasın) ---
STATE_DIR = f'/content/btc_puzzle{PUZZLE_BITS}' if os.path.isdir('/content') else os.path.abspath(f'btc_puzzle{PUZZLE_BITS}')
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        STATE_DIR = f'/content/drive/MyDrive/btc_puzzle{PUZZLE_BITS}'
    except Exception as e:
        print('⚠️  Google Drive bağlanamadı, yerel klasör kullanılacak:', e)
os.makedirs(STATE_DIR, exist_ok=True)
STATS_PATH  = os.path.join(STATE_DIR, 'stats.json')
BLOCKS_PATH = os.path.join(STATE_DIR, f'blocks_done_b{BLOCK_BITS}.txt')
LOG_PATH    = os.path.join(STATE_DIR, 'log.txt')
FOUND_PATHS = sorted({os.path.join(STATE_DIR, 'FOUND_KEY.txt'), os.path.abspath(f'FOUND_KEY_puzzle{PUZZLE_BITS}.txt')})
VS_OUT      = os.path.join(STATE_DIR, 'vanitysearch_found.txt')
print(f'Kayıt  : {STATE_DIR}')

def log(msg):
    line = f'[{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S} UTC] {msg}'
    print(line)
    with open(LOG_PATH, 'a') as f: f.write(line + '\n')

def load_stats():
    try:
        with open(STATS_PATH) as f: return json.load(f)
    except Exception:
        return {'total_keys': 0, 'seconds': 0.0, 'sessions': 0}

def save_stats(st):
    tmp = STATS_PATH + '.tmp'
    with open(tmp, 'w') as f: json.dump(st, f, indent=1)
    os.replace(tmp, STATS_PATH)

def load_done_blocks():
    try:
        with open(BLOCKS_PATH) as f: return {int(x, 16) for x in f.read().split()}
    except FileNotFoundError:
        return set()

def mark_block_done(idx):
    with open(BLOCKS_PATH, 'a') as f: f.write(f'{idx:x}\n')

def on_found(cands, source=''):
    """GPU aracının bildirdiği aday anahtar(lar)ı bağımsız doğrular; doğruysa her yere yazar."""
    global FOUND_KEY
    k = next((c for c in cands if RANGE_START <= c <= RANGE_END and addr_from_key(c) == TARGET_ADDRESS), None)
    if k is None:
        log(f"⚠️  {source}: bildirilen anahtar(lar) {', '.join(f'{c:x}' for c in cands)} hedefle EŞLEŞMEDİ (yok sayıldı).")
        return False
    FOUND_KEY = k
    wif = wif_from_key(k)
    txt = (f'PUZZLE {PUZZLE_BITS} COZULDU  ({datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S} UTC, {source})\n'
           f'Address          : {TARGET_ADDRESS}\n'
           f'Private key (HEX): {k:064x}\n'
           f'WIF (compressed) : {wif}\n'
           f'Public key       : {pubkey_compressed(k).hex()}\n')
    for path in FOUND_PATHS:
        try:
            with open(path, 'a') as f:
                f.write(txt + '\n'); f.flush(); os.fsync(f.fileno())
        except Exception as e:
            print('yazılamadı', path, e)
    log('🎉🎉🎉 ANAHTAR BULUNDU VE DOĞRULANDI! 🎉🎉🎉')
    print('=' * 78 + '\n' + txt + '=' * 78)
    print('Kaydedildi:', ', '.join(FOUND_PATHS))
    display(HTML(f'<div style="font-size:22px;padding:16px;border:4px solid #1a7f37;border-radius:8px">'
                 f'🎉 <b>BULUNDU!</b><br><code>{k:064x}</code><br>WIF: <code>{wif}</code><br>'
                 f'<b>Fonları açık mempool üzerinden gönderme</b>, en alttaki talimatları oku.</div>'))
    display(Javascript('''(async()=>{const c=new AudioContext();for(let i=0;i<20;i++){const o=c.createOscillator();
        o.frequency.value=880;o.connect(c.destination);o.start(c.currentTime+i*0.5);o.stop(c.currentTime+i*0.5+0.25);}})()'''))
    return True

# Daha önceki bir oturumda bulunmuş mu?
for p in FOUND_PATHS:
    if os.path.exists(p):
        print(f'‼️  {p} zaten var, içeriğine bak:\n' + open(p).read())

_st = load_stats()
print(f'Önceki oturumlar: {_st["sessions"]} oturum, {fmt_num(_st["total_keys"])} anahtar, {fmt_dur(_st["seconds"])}')

In [ ]:
#@title 🔧 4) CUDA derleyicisi + VanitySearch-Bitcrack derleme (GPU'ya özel, ~2-5 dk)
WORK = '/content/tools' if os.path.isdir('/content') else os.path.abspath('tools')
os.makedirs(WORK, exist_ok=True)
sh('command -v git make g++ >/dev/null || (apt-get update -qq && apt-get install -y -qq git make g++)', quiet=True)

def nvcc_archs(nvcc):
    try:
        out = subprocess.run([nvcc, '--list-gpu-arch'], capture_output=True, text=True, timeout=60).stdout
    except Exception:
        return set()
    return {int(x) for x in re.findall(r'compute_(\d+)', out)}

def pick_nvcc():
    seen, best = set(), (None, set())
    for n in [shutil.which('nvcc'), '/usr/local/cuda/bin/nvcc'] + sorted(glob.glob('/usr/local/cuda-*/bin/nvcc'), reverse=True):
        if not n or not os.path.exists(n) or os.path.realpath(n) in seen: continue
        seen.add(os.path.realpath(n))
        a = nvcc_archs(n)
        ver = re.search(r'release ([\d.]+)', sh(f'{n} --version', check=False, quiet=True))
        print(f'  nvcc {n}  (CUDA {ver.group(1) if ver else "?"}, en yüksek arch: {max(a) if a else "?"})')
        if CC in a: return n, a
        if a and (not best[1] or max(a) > max(best[1])): best = (n, a)
    return best

def install_cuda_nvcc(ver='12-8'):
    # NVIDIA'nın resmi apt deposundan sadece derleyici + runtime başlıkları (tam toolkit değil)
    sh('apt-get update -qq', check=False, quiet=True)
    pol = sh(f'apt-cache policy cuda-nvcc-{ver}', check=False, quiet=True)
    if not pol.strip() or 'Candidate: (none)' in pol:
        osid = sh('. /etc/os-release && echo ${ID}${VERSION_ID//./}', quiet=True).strip()
        sh(f'wget -q https://developer.download.nvidia.com/compute/cuda/repos/{osid}/x86_64/cuda-keyring_1.1-1_all.deb '
           f'-O /tmp/cuda-keyring.deb && dpkg -i /tmp/cuda-keyring.deb && apt-get update -qq', quiet=True)
    sh(f'DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cuda-nvcc-{ver} cuda-cudart-dev-{ver}', quiet=True)

print('CUDA derleyicileri:')
NVCC, ARCHS = pick_nvcc()
if CC not in ARCHS and AUTO_INSTALL_CUDA:
    print(f'→ Mevcut nvcc sm_{CC} için yerel kod üretemiyor; CUDA 12.8 nvcc kuruluyor...')
    try:
        install_cuda_nvcc()
        NVCC, ARCHS = pick_nvcc()
    except Exception as e:
        print('⚠️  CUDA 12.8 kurulamadı, PTX (JIT) yedeğine geçiliyor:', e)
if not NVCC:
    raise RuntimeError('nvcc bulunamadı. Bu notebook NVIDIA GPU + CUDA toolkit olan bir Linux ortamı ister.')
CUDA_ROOT = os.path.dirname(os.path.dirname(os.path.realpath(NVCC)))

if CC in ARCHS:
    GEN_VS = f'-gencode=arch=compute_{CC},code=sm_{CC} -gencode=arch=compute_{CC},code=compute_{CC}'
    print(f'✅ Yerel derleme: sm_{CC}  ({NVCC})')
else:
    ptx = max(a for a in ARCHS if a <= CC)
    GEN_VS = f'-gencode=arch=compute_{ptx},code=compute_{ptx}'
    print(f'⚠️  sm_{CC} desteklenmiyor → compute_{ptx} PTX üretilecek, sürücü ilk çalıştırmada JIT derleyecek.')

VS_DIR = f'{WORK}/VanitySearch-Bitcrack'
VS_BIN = f'{VS_DIR}/vanitysearch'
JOBS = os.cpu_count() or 2

VS_CAND_PATCH = 'fprintf(stdout, "  CandKey: %s\\n", k.GetBase16().c_str());'

# GPU hız yaması (gzip+base64; okunabilir hali: vanitysearch-opt.patch). İçerik (sm_120 ölçümleri):
#   512->320 indirgemesinde kaydır/topla yerine gerçek çarpma (ALU -> FMA hattı)      +%5.0 hız
#   tek hedef: hash160 sabit belleğe karşı doğrudan; hash/kontrol satır içi           +%1.9 hız
#   SHA-256/RIPEMD-160 toplamaları IADD3 yerine IMAD*1 (doymuş ALU -> boş FMA hattı)  -%2.1 çevrim
#   128 thread/blok, 2 kat blok (aynı thread sayısı, daha ucuz bariyer)               -%2.1 çevrim
#   toplam: orijinale göre -%10.8 çevrim/anahtar (saat limitli kartta ≈ +%12); RTX 5050 laptop +%9.0
VS_OPT_PATCH_B64 = (
    'H4sIAAAAAAAA/807bXPautKfm1+xJ3fasYNDLBkbU046BQxpbkmTCc15XnJyGYNF8I2xGds00HN6fvszK8kvOCRNT9s7D50GLO2u'
    'VrurlXZX9vzZDA4Pb/0U3KOTiyv834sWy1XK6nOYPGja80OPrcFt2sR0ab0+mbYYcz0gum41GnuHh4c76OzVarVdtN6+hUNqak2o'
    'UVMjFrx9uwf83+F47LFP/pSNxzAeh5EfBn6ID58i34PenE3vLiI/TJWVH6YGHadwMJ5rkD344TTWwPW8mCUJdsqfGhTwQRTdrZYG'
    'LbdFq1SFP/ZqR0cw8sPbgMFsFQSHEhsS5sbT+WtI5wzmbjInlg5+AtNosXRj5oHnx2yaBhtwb10/TFKYRmGSumEKC7aI4g0njB0o'
    'sGgGt5Ef3kI6j6PV7ZxTJdR+34VlzGb+GlJ3EjBQopDBbRBN3ACCyPVgyWK4Yxu1vlfbklI22HhczGic8Gl8dONblsIx6O3nIKUc'
    '/B2x9Gvzpr1X26v9w5+FHptB712/9358+mF4+qG/V/uHx2Z+yLZaYTyeRfGUZRrbq/2DhZ4/QyqlkbdQ/iNKhT2AohUgms3a2CSp'
    'BNgYGLTNjdKwNQtqjaZGKLx9Cz/aGiusBAufs7IHezUAAH8GypbmuFFiT947v9Zv4JfjLV3pNyrELF3FYfsBNHkATW7gzz9hPL+m'
    'D7po1mU86DKyrsaDrsau4W+jNMKpn6ZsIZu/7NVwsgBHRzDk+hKWLhqXsQ7HcKCUxKUq47nKBQQw99GMZef1MtZv2nveDhfWD2/9'
    'kNWnq8Lt5E3Shc0mJiXmtF633Jk984ydLqxA2nJhRTNaS7OpER1qzaZmGGgufpjCuBeFn1icjs5+YzHtRTFLFGxfuP+OYm4bsPDD'
    'KJamiT4vW04nlxfj0en/9oHotFGssrz5t9H45PJiD7KO0ce+7MlADojwocJtZIaKfmp8xzZJIdwDSDrVVXQA26vIamAb4pVhSq5y'
    '6K7C6Ryd3cy/XcVu6kehBtEnFse+x32Ym8Jk5QcepP6CgXLo/DYaf7zoHtfrdWDptK5yOoTa6AyZ6yVHkyCa3oEycePYZ3GCzjGE'
    'ZBNOoQH3brxMyn7UVuHeT+dA19yJctwE7hhbckROXBLm3jMQDOMfN7xlXhsOaZ28hOlmGjABcsc2EIWQLMaE6qBMWDidH4n++pL7'
    '3swliqkUahLPOJnC9ZVguxejsy1gbABqWlvQEvjs9EN3eN57P9rGOL/64IxgPBbzGE+iVegl47Eihta2MFWkGyRsB4Wyaz46gi6K'
    '7fDe9xjkYvfDwmq4QLmAhQKiGbhC1igpbE/cBYOT9bV/c3SyufZv8v0vqfMRPkSQzPlOKfZD3DxXCfM0SCIksAE3ZhBGKYSMecyD'
    'WRTDNIpxVw1xA8YZfjjvdi4vT/uXxwS8OBIqXkiNgJijtwWowqtXUG7YksXofz70lJ1SEj0wHqPdSfsRkMWGtr3CCuX8jLXG3QR+'
    'CsgkdeN0DcccBWqgcH2ceuv6Gg6Echx/UV+rcAC22NqIbuCBi1BLa4q97cd6iW0WIVlNlteZXzqiN9eNG+nK0UXhd0W+cpfI5S+h'
    'h5HrUdPqKMlak9N+2LWRXRvsErMlDT5bo/kzZxt53ni5gWNINrgvvwIiWRufsPSdOCniqZczL4A14KcFe5weqPnuVjpKzLXC4R8B'
    '1UpMFaygYbSfK0UxxFnkjVYTalpKsg41GNOTEAVaFaaypTfkAA6B3CBkqEpDMohGGlAjZgO/f5pwfdxHE4YnDn6AFYI9i7zT8JMi'
    'O/n8vsGacll8YLd8tptQg4SbTaUDhSSkI7rQJSk+5wR8+BWqImqDX6uJ1cr5eZynXXxVNTS+XWvCo2aSl2xkMjhbBanioQZRXwhW'
    'SIQryWxx+7fpz7L/gp+jB59vF8IuMXi4rFFBfFMpi6qQwEYDb53N2dZ5SMkd3U+c85fn29zesxQmXVaLq4zSn6yyx9X17b4E5Y+u'
    'ZFNZRVl32YqllqjwzPTneuZc5OhxvR2sLTPO1TZsf77No4zSKGZiF+KbkwbL9SN9Gw2WxQ6FArChRi1ds1EOeXTx+nX+k0cOt8vV'
    'qVcKbBfueoAnP+5s8DNdea7D49SLOFqCl/9sF/0nLC1AWJz6LFFeFZCaGKVwpx+644+X7/odZ3zRvxyfXJ5fXcAxHler0nri89DQ'
    'AE4uTx0QfrNqhHxcnHA4+ciFfhJHqyUclyZUX6yC1L+IoylLkijuRaswhQM8c0vt7GZbnI9/Hue1v825jAPylYVR+y+xG3rR4izy'
    'WK7iraPVMrpn8fns430Ex3jiQGsyCcHMhWk0NMpjUb6WSjY1YqlcQ0qSeq9ff2LTNIp/HXYc57I/Gr3JYmtWzqukUeoGAg05RMdX'
    'm0RRUCE8KqUsFH72h+yMk+XLeMwoBJVTn4ss01YbJr34pHjzgi2myw2eiSQZDaierUQ06zMO8DEabRaTKFBKaQkN5hok/mcWzZS5'
    '+hROOeGiwasoZDliFDKOWiD34ziKxymwOIbjbGUN3STlHbmXQD0iyC8CZrSaot630jnL2A/TmbJfiBIqonwNL5Pfw30tG4YPMUpj'
    'P7xF4vmk8CMSMDBzg4QV+ZZSTxqvsGOvxtMwaKxlh8PSUcqWI/8zU/KcWY6a5xmeTLmUksZZi0y42IwRyyL1OiFNvaHbTyVcKinj'
    'vBVt3GqgiVu4fbx9C8vVJPCnr5FTbuuDmLHCcQoXzDsesfx8dymZ/texdq2XXcsFSdUA+GJ5coXAQbZC+NgZwnvcAHlEAAfL7a5L'
    '7h7+uVosJcBym+0Rz1aj++Dbh0hec2+yW3/cWZXVJxuyjL9p6SZt1eu6ZXqmO9upvQxlS3lZo4jHcLMjRKOYWN2Do6PcnfFdWJzy'
    '/wB93dQNQ7csa9Ds2gbRjavhUMPmVr9L+q1Wy+qZVr/ZzZsb3W6r1TDsDqWWaclmajcMm1rNZstotvq0fzUcwhfuZXlCAocU+8LJ'
    '5cXBCSi33GHjqsfchCuyVoc8axUtRV4LEx0n66OTjchaJjBhQXQPU8x3AeYyI563U7eyPpivK6cWTi4vsuxekfTJOo6PRR88lanP'
    'RbaWItPXjtlt6aRL+9Q2iUEGKAN93en0e1a/5/RsYnR12xatjmU1OsQeWFbHsWhDF620QQb9btfuU6PXdZrNq+HwSxuOjr6BnUyD'
    '+rrT7Rp9yxrQpqnr1HLEED3H1AeOPug6eq9rdiSTjtVrUL3rEKNlE2cgmTSJYTRtpzUYtBoD20ET+NLmmZotYZmEPl3VeCAr0Nct'
    'avVpT9f7A6PRoQ3Jnd5xur2WZTutPjFbjhRLSzc7ttnstHqDFrEdU7Q2LNNo6l1qNzvN1mBgSNt6HiclQ+/QgT3oUr1rGLbdHDTk'
    '3G27r7e6lJiO0ewQyUmn0RnYXaczcPr9Hu0R0WqYfdMgXcM2LLun2zTjpCoonmX8VkHpVsPqU2PgmAOTmDpXDehrx+4ZrV6n65id'
    'Hul1OCegr/sd2rH6Rp80qdOnRgZtU5tSy6CE9iy95bT+rqwcg3StfmcwsPrU6gnrAX1tUcciRqdHB80uaXb7stns29S0+rahdy29'
    '08uaCRnYHVtv2abZdAb9RklcPAPIcH+FfSm3xSpJYcJQeBqaGkQxX577+doFdGPPFmv72dA4792uGlNKZU8tnqWj1luz5rSl1+u2'
    '6bao6e501BJjy0/LNnTTDX6KbBCN6g/qX7tKhu+vb+B4T7rWzM9dnl8qay1UQVHWb96E6p/K+tdfFYMehqqqyqTzeZ5jP+04jgGK'
    'ceiHy1WK26oKURhgkjhZsSTLMXeGV4dz5n7awNJfMg3u5/50DqN3g6Ph+YUBboDnblFpTdwUqxECKGBwetZxIF6FOa3BWYdTgXTu'
    'pujw/7L0l+B7AavDeOZ6nuJqExXrTQekNkEAPJYimbasKTAgGKiyBGZxtKjWe2Vam4eyfsBimLohZrdnUeCBn8LEnd7hESwCNxTz'
    'F3ny30bjQcdxYOKnyWsgcAyjdx1qWhBjvIeiSTSgRfOCJYl7yyCZzpm3CpgGDTiGy9OL/pmDNeoCbbtugYNsbUt81GaxKX29VFyc'
    '07dgt0rAJXAu1PzRLYUYE3kozhtifnByk4Wyv3C9ehDVVwaFl7oGL4kGL6kGL432PryG/eN4X4lV/BXvK66q8W/kTP6cyBOyPMYi'
    '4S+l6pmurFVQhLFSFf4l7ZYYxW9KVbWEQUoYVgmDlDDMMkZSHqNZwrDxt7KGN2/AUFURwZm8mGi1NNL8lrWXF+PnylrbaJ9VUD4L'
    '4q9A2cC/4DOuOn7ykWcNru1XQAoTGNFLNBXF1WCiwVQDTwOmwUyDWx5H3Wlwr8LvIqBI0TCFRkdEQWGLh95cYdpMu1U1mEMN7rAs'
    'ca+qajtD9KB2DCnJn+cFHZ3rLyVQgzP337j8tCnXnvDN8K2MQs4ossLZhBrkHBbstTPgEnOHGXOcH87bFl/tEv+PgZQ3ia9rcny/'
    '8NdjP/RTodM/uEXYJjpjolsaxSsycH9NzBtkMyHK/TUxbnDM+2v7Bmpoafd4EaD9uwjTscj+X2en/11VOi2Ujt0Y7aE6EDfXhiDf'
    'uMk1y4kTfOZwOGjrRuVywPYKollBpAKRCESi55i0gqlXEA2BSCUiyRGN6pAVRM46B+OINEdsVBBpBZGzzsE4opEjmhVEo4JoCURT'
    'IjZyRKuCWJVqUyBaEtHMEZsVxKpUbYHYFIiFUO0KnlXBawk8bjRcchKvVcFrVtXPtcPhELGQKalajl3FlJaDgIhaSJVUbadVRc1s'
    'RxpPIVdSNR7BXxlXmg9CIm4hWvLAfqoGJIxfQCKuVeBWTUiwWMaVRoSQiNsscKtWJFgs4UoZ82XOlSRQvzzwg6W1K9dk5hXwNo5Y'
    'oJlXIMIriBWYgZkSTCglW6ft3/NSMXqehky1SEFkuETicqPN5SXHaJQAqQTkRpoLRwI+05VtHbbhEqvspe2An4PGlx8c5U6FPyT3'
    'rZZGDKjRXcdYniTJT0inoZ/6boBpr9wZJ2gpmAPLBpk1lLUGGw1wa+Xb6mcV/gTlr8/wCjbl7X5mliDXuAlv4E/46zPffyuOuHTB'
    '5/L0ouM4DdzUZhqsNbhTpVHMNHChBmvcr9QqgpEhVKEfXmrYNYKCsLPnUC8gi0sQxWH/ItuRJ9pU8zTcYNfanRbjNnwIsMJjdHkk'
    'uf1j+w625H7sYu/5UFlpEKNpMNk+Fe1EV3CPhQdsjO8qjOxk4wELxWS/cXweM+lYsKs1TKIR8qS5fYzdMJlF8WLb2vIE4gEeXnj2'
    'lZcYn7x9WL1hkEeQB2stzy36ybknS0d5qpGft7Ng4F1n9O7Btc5S4zNvdZYxvp+77RubB2sDQ57SHUtVkRW2vEnkgt+zTXeTsuSa'
    'WI/Ez2duuhU/i2cZP7d06k3sRr0+neizyWR3olNibMXPso0X9fhZrSa+MM+ZybR7ijkLXj5q6NKZdZKELSbBRt4f9j+xpLgMeNVx'
    'nHNlqmGwNFExHoJPUeCmGNAq+67n1afT+spqlCIjGRUF+8qUR0WBjIoCDIWwQUSn+4B3R8rj9J4aZ/p9A9V+1oSqtH/oJNC+ysSf'
    'pv2NhCtKJpz4VyXC6QvatW3aT6r2cerTv0e+9mOZf6DG72e4qrynKD6TXEmoo6vu42acrCY/al2OrrqPm3SymvywdfmDJ1Sl/UMn'
    'saXa0VX3adp/f12iTB4xm6pE/sa6RKk8Tv271+UPYf6BGr+f4arynqL4nPlvnwCvzq6Gw3MliLSdBrFYBTyX+IhJBNFXjO3savju'
    'VJn7j1Of+49Sn/s7qecWccb3xlgTyaMHxF0PiT9YKaVMaJBlQreGEN/TRw0Rh+09Nez0h45b+w9NtzrUz5xi2UZwrK8O9R3jZPcE'
    'KX9JCb9Eppg7wvi6caPxaiH+UtsiCyjr3XgTzl8GG5hssA6m48dwCL/UioUKk9DDNwbVIWbeaor1bkhStqzDFWKOlmzquwGw9dIN'
    'vQQrGFi8ELWWuT9LkyOsMYDSGV7xuoraxkD3su+cXQ2PCV71TyBmbgD8EtQy8FmSlUjuo/gObiOWYP28XJvZrlgIWls1C9HEs9jb'
    'pfQq7CXOifHLgPg6liomxX9rZWlcDYc7IuedyFIioqkUE+Gp/vDnfMquDu9qS3ZkzsOwGni/wmjiS5U7QtD8OmQRiMXFvUk4cMsP'
    'vCQDIM9OsUnodfNGg5TblYjOMBXDJSPqX1geTSMwqI6x9tGRkHCqAceGGjTUB6Lm99e3pLkFL0o3JfFXeyWD55JDnXOIaaOspyd7'
    '8PZ6yrNg1R7KezDvJYuf+GpErVG8IfGNQvxOuf2/EcZW6PybG/rppj5dLmFSepBBM9FnljWd1evEdSfMqrwPXAIXEXOpASXeIha6'
    'MvwSbxKIt2k4kLjG9Pr1wA+992xzcnGlfHw3vuhcds4OYCkSBS9u6/mFp4+bJVOS/CdO8AVew8Na8mAVBAJBYGQ3u/AlJPl7qEE4'
    'Kd3desGR8yZ+OYenTl68eCHeBQhTKL0PgM9qfiesjleJFDV/KQCxOMEc4tq/qfspW4hLgdgtJ1O+KaZk77qW4a/1m3pxZ0ygTmLm'
    '3smHL/yL//2yB/w/+rSnp6+29/4PD5RlyxI+AAA='
)
VS_OPTIMIZED = False

def build_vs(cxx='g++', optimize=None):
    global VS_OPTIMIZED
    if optimize is None: optimize = VS_OPTIMIZE
    if not os.path.isdir(VS_DIR):
        sh(f'git clone --depth 1 https://github.com/FixedPaul/VanitySearch-Bitcrack.git {VS_DIR}', quiet=True)
    sh(f'git -C {VS_DIR} checkout -- .', quiet=True)          # önceki yamaları sıfırla, her derleme temiz başlasın
    VS_OPTIMIZED = False
    if optimize:
        import base64, gzip
        with open('/tmp/vanitysearch-opt.patch', 'wb') as f:
            f.write(gzip.decompress(base64.b64decode(''.join(VS_OPT_PATCH_B64))))
        if subprocess.run(['git', '-C', VS_DIR, 'apply', '/tmp/vanitysearch-opt.patch']).returncode == 0:
            VS_OPTIMIZED = True
        else:
            print('⚠️  Optimize yaması bu kaynak sürümüne uygulanamadı (upstream değişmiş olabilir); orijinal kaynak derleniyor.')
    mk_path = f'{VS_DIR}/Makefile'
    mk = open(mk_path).read()
    # Makefile'daki sabit sm_60..sm_89 listesini bu GPU'nun mimarisiyle değiştir
    mk2 = re.sub(r'(?:-gencode=arch=compute_\d+,code=(?:sm|compute)_\d+\s*)+', '$(GENCODE) ', mk)
    if mk2 != mk:
        with open(mk_path, 'w') as f: f.write(mk2)
    # Aracın CPU tarafı (SHA-256/checksum) yeni GCC'lerle hatalı derlenebiliyor ve GPU'nun bulduğu DOĞRU anahtarı
    # "wrong private key" diye reddediyor. Yama: o durumda da aday anahtarı yazdır; doğrulamayı Python yapar.
    vc_path = f'{VS_DIR}/Vanity.cpp'
    vc = open(vc_path).read()
    marker = 'fprintf(stdout, "  Endo:%d incr:%d comp:%d\\n", endomorphism, incr, mode);'
    if VS_CAND_PATCH not in vc:
        assert marker in vc, 'Vanity.cpp beklenen yerde değişmiş; yama uygulanamadı'
        with open(vc_path, 'w') as f: f.write(vc.replace(marker, marker + '\n\t\t\t' + VS_CAND_PATCH, 1))
    if cxx != 'g++' and not shutil.which(cxx):
        sh(f'apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {cxx}', quiet=True)
    sh(f'make -C {VS_DIR} clean; rm -f {VS_BIN}', check=False, quiet=True)
    sh(f'make -C {VS_DIR} -j{JOBS} all CUDA={CUDA_ROOT} NVCC={NVCC} CXX={cxx} CXXCUDA={shutil.which(cxx)} GENCODE="{GEN_VS}" '
       f'CXXFLAGS="-mssse3 -Wno-write-strings -O2 -fno-strict-aliasing -I. -I{CUDA_ROOT}/include -include cstdint" '
       f'LFLAGS="-lpthread -L{CUDA_ROOT}/lib64 -lcudart_static -ldl -lrt"', quiet=True, tail=8000)
    assert os.path.exists(VS_BIN), 'vanitysearch ikilisi oluşmadı'
    print('  ' + (sh(f'{VS_BIN} -v', check=False, quiet=True).strip().splitlines() or ['?'])[-1] +
          ('  [optimize yaması uygulandı]' if VS_OPTIMIZED else ''))

t0 = time.time()
print('\n🔨 VanitySearch-Bitcrack derleniyor...')
build_vs()
print(f'✅ VanitySearch-Bitcrack hazır ({time.time() - t0:.0f} s)')


In [ ]:
#@title 🧪 5) Öz-test: araç bu GPU'da gerçekten anahtar buluyor mu?
#@markdown Rastgele bir test anahtarı üretip adresini Python'da hesaplıyoruz, sonra GPU aracına 2^36'lık küçük bir aralıkta aratıyoruz.
#@markdown Test geçmezse arama **başlamaz** (Blackwell'de bozuk çalışan bir araçla saatler yakmayalım).

def selftest(mode, bits=36, timeout=300):
    base = ((1 << 23) | secrets.randbelow(1 << 23)) << bits     # 2^bits'e hizalı başlangıç
    k = base + secrets.randbelow(1 << bits)
    addr = addr_from_key(k)
    cmd = [VS_BIN, '-gpuId', '0', '-start', f'{base:x}', '-range', str(bits), '-stop'] + \
          (['-random'] if mode == 'native' else []) + [addr]
    t0, ok, own_check_broken, tail = time.time(), False, False, []
    for line in stream(cmd, cwd=VS_DIR, timeout=timeout):
        if not line: continue
        tail = (tail + [line])[-12:]
        if 'wrong private key' in line: own_check_broken = True
        ev = parse_line(line)
        if ev and ev[0] == 'keys' and k in ev[1]:
            ok = True; break
        if mode == 'blocks' and ('Range Finished' in line or 'KEY NOT FOUND' in line):
            break                                   # aralık bitti ama anahtar gelmedi
    print(f'  {mode}: {"✅ GEÇTİ" if ok else "❌ BAŞARISIZ"}  ({time.time() - t0:.1f} s, test anahtarı {k:x})')
    if ok and own_check_broken:
        print('    (aracın kendi CPU adres kontrolü hatalı; anahtar GPU sonucundan alınıp Python ile doğrulandı)')
    if not ok:
        print('    son çıktı:\n    ' + '\n    '.join(tail))
    return ok

SELFTEST = {'blocks': selftest('blocks', timeout=600)}   # ilk çalıştırma PTX-JIT ise uzun sürebilir
if not SELFTEST['blocks'] and VS_OPTIMIZED:
    print('  → Optimize yamalı sürüm testi geçemedi; orijinal kaynakla yeniden derleniyor...')
    build_vs(optimize=False)
    SELFTEST['blocks'] = selftest('blocks')
if not SELFTEST['blocks']:
    print('  → VanitySearch, yazarın Makefile\'ında sabitlediği g++-9 ile yeniden derleniyor...')
    try:
        build_vs('g++-9', optimize=False)
        SELFTEST['blocks'] = selftest('blocks')
    except Exception as e:
        print('  g++-9 ile derlenemedi:', e)
if not SELFTEST['blocks']:
    raise RuntimeError('VanitySearch öz-testi geçemedi; bu GPU/derleme ile arama yapmak anlamsız.')

STRATEGY = 'blocks'
if VS_STRATEGY == 'native':
    if selftest('native'):
        STRATEGY = 'native'
    else:
        print('ℹ️  VanitySearch -random öz-testi geçmedi → rastgele-blok modunda kullanılacak.')
print(f"\n🏁 Strateji: {STRATEGY}  ·  {'optimize yamalı' if VS_OPTIMIZED else 'orijinal'} VanitySearch")


In [ ]:
#@title 🚀 6) ARAMA — durdurmak için ■ Stop. Tekrar çalıştırınca istatistikler Drive'dan devam eder.
if FOUND_KEY is not None and addr_from_key(FOUND_KEY) != TARGET_ADDRESS:
    FOUND_KEY = None      # önceki (ör. test) bulmacadan kalan sonuç; bu hedefle ilgisi yok

def build_cmd(start=None, bits=None):
    if STRATEGY == 'native':
        rng = ['-start', f'{RANGE_START:x}', '-range', str(RANGE_BITS), '-random']
    else:
        rng = ['-start', f'{start:x}', '-range', str(bits)]
    return [VS_BIN, '-gpuId', '0', '-o', VS_OUT] + rng + ['-stop', TARGET_ADDRESS], VS_DIR

stats = load_stats()
stats['sessions'] += 1
done_blocks = load_done_blocks()
S = {'t0': time.time(), 'keys': 0, 'speed': 0.0, 'last': '', 'blocks': 0, 'restarts': 0, 'gpu': ''}
status = display(HTML('<i>başlıyor…</i>'), display_id=True)

def gpu_telemetry():
    q = sh('nvidia-smi --query-gpu=utilization.gpu,power.draw,temperature.gpu,clocks.sm --format=csv,noheader,nounits',
           check=False, quiet=True).strip().split(',')
    return f'%{q[0].strip()} kullanım · {q[1].strip()} W · {q[2].strip()} °C · {q[3].strip()} MHz' if len(q) == 4 else ''

def render():
    el = time.time() - S['t0']
    sp = S['speed'] * 1e6
    tot = stats['total_keys']
    frac = tot / RANGE_SIZE
    rows = [
        ('Motor', f"VanitySearch{'-opt' if VS_OPTIMIZED else ''} / {STRATEGY}"),
        ('Hız', f'<b>{sp / 1e9:.3f} Gkey/s</b>'),
        ('GPU', S['gpu']),
        ('Bu oturum', f"{fmt_num(S['keys'])} anahtar · {fmt_dur(el)} · {S['blocks']} blok bitti · {S['restarts']} yeniden başlatma"),
        ('Tüm oturumlar', f"{fmt_num(tot)} anahtar · {stats['sessions']} oturum · {fmt_dur(stats['seconds'] + time.time() - last_save)}"),
        ('Taranan oran', f'{frac:.3e}  (≈ 1 / {1 / frac:,.0f})' if frac else '0'),
        ('Günlük şans', f'≈ 1 / {RANGE_SIZE / (sp * 86400):,.0f}' if sp else '-'),
        ('Son satır', f"<code>{S['last'][:140]}</code>"),
    ]
    status.update(HTML('<table style="font-family:monospace;font-size:13px">' +
                       ''.join(f'<tr><td style="padding-right:14px"><b>{a}</b></td><td>{b}</td></tr>' for a, b in rows) +
                       '</table>'))

def scan_vs_out():
    # VanitySearch bulduğunu -o dosyasına da yazar; ekrandan kaçırılırsa buradan yakalarız
    if not os.path.exists(VS_OUT): return False
    for line in open(VS_OUT, errors='replace'):
        ev = parse_line(line)
        if ev and ev[0] == 'keys' and on_found(ev[1], 'vanitysearch -o dosyası'):
            return True
    return False

def pick_block():
    nblocks = RANGE_SIZE >> BLOCK_BITS
    while True:
        idx = secrets.randbelow(nblocks)
        if idx not in done_blocks: return idx

last_save = last_render = last_tel = time.time()
log(f"Oturum başladı: VanitySearch{'-opt' if VS_OPTIMIZED else ''}/{STRATEGY}")
try:
    scan_vs_out()
    while FOUND_KEY is None:
        idx = None
        if STRATEGY == 'blocks':
            idx = pick_block()
            cmd, cwd = build_cmd(RANGE_START + (idx << BLOCK_BITS), BLOCK_BITS)
        else:
            cmd, cwd = build_cmd()
        run_keys, completed, last_progress, info = 0, False, time.time(), {}
        for line in stream(cmd, cwd=cwd, info=info):
            now = time.time()
            if line:
                S['last'] = line
                ev = parse_line(line)
                if ev and ev[0] == 'keys':
                    if on_found(ev[1], os.path.basename(cmd[0])): break
                elif ev and ev[0] == 'speed':
                    delta = max(0, ev[2] - run_keys); run_keys = ev[2]
                    S['keys'] += delta; stats['total_keys'] += delta
                    if 0 < ev[1] < 1e7: S['speed'] = ev[1]
                    last_progress = now
                elif 'Range Finished' in line or 'KEY NOT FOUND' in line:
                    completed = True
                elif re.search(r'error|wrong|fail|warning', line, re.I):
                    log(f'araç: {line}')
            if now - last_progress > 180:
                log('⚠️  180 sn ilerleme yok, araç yeniden başlatılıyor.'); break
            if now - last_tel > 15: S['gpu'] = gpu_telemetry(); last_tel = now
            if now - last_render > 3: render(); last_render = now
            if now - last_save > 60:
                stats['seconds'] += now - last_save; save_stats(stats); last_save = now
        if FOUND_KEY is not None or scan_vs_out(): break
        if idx is not None and completed:
            done_blocks.add(idx); mark_block_done(idx); S['blocks'] += 1
        elif not completed:
            S['restarts'] += 1
            log(f"araç kapandı (rc={info.get('rc')}), 5 sn sonra yeniden başlıyor.")
            time.sleep(5)
except KeyboardInterrupt:
    log('⏹️  Kullanıcı durdurdu.')
finally:
    stats['seconds'] += time.time() - last_save; last_save = time.time()
    save_stats(stats); render()
    if FOUND_KEY is not None:
        print(f'\n🎉 ANAHTAR: {FOUND_KEY:064x}\n   Dosyalar: {FOUND_PATHS}\n   Aşağıdaki "Bulursan" bölümünü OKU.')

## 🎉 Bulursan ne yapmalı (ÇOK ÖNEMLİ)

**Fonları normal bir cüzdandan, açık mempool üzerinden göndermeye çalışma.** Bu adresten harcama yaptığın anda public key mempool'da görünür. Anahtar yalnızca 70 bitlik bir aralıkta olduğu için botlar Kangaroo ile özel anahtarı **saniyeler içinde** bulur. Sonra daha yüksek ücretli bir işlemle (RBF) senin işleminin yerine geçip fonları kendilerine çekerler. Bu olay 2024'te #66 bulmacasını çözen kişinin başına geldi.

1. Önce `FOUND_KEY.txt` dosyasını **çevrimdışı** bir yere yedekle: kağıda yaz, şifreli bir USB'ye kopyala.
2. İşlemi çevrimdışı bir makinede oluştur ve imzala. Örneğin Electrum'un offline modu ya da kendi kodun olabilir.
3. İşlemi **doğrudan bir madenciye özel kanaldan** gönder. Örnek: MARA Slipstream gibi, işlemi açık mempool'a koymadan bloğa ekleyen servisler. Böylece public key, işlem bir bloğa girene kadar kimseye görünmez.
4. Fonlar güvendeki adresine geçtikten sonra Drive'daki `FOUND_KEY.txt`'i ve notebook çıktısını sil. Bu çıktıyı kimseyle paylaşma.

### Notlar
* **Oturum koparsa:** Bulunan anahtar zaten Drive'a yazıldı. Arama hücresini (6) tekrar çalıştırman yeterli; toplam istatistik Drive'daki `stats.json`'dan devam eder.
* **`blocks` stratejisi:** Tamamlanan bloklar `blocks_done_b42.txt`'e yazılır ve bir daha taranmaz. `native` modda VanitySearch'ün kendi rastgeleliği kullanılır. 2^70'lik bir alanda tekrar tarama olasılığı ihmal edilebilir düzeydedir.
* **Tüm GPU gücü:** VanitySearch bütün SM'leri kullanır. Arama hücresinin altında anlık kullanım, güç, sıcaklık ve saat değerleri görünür.